- công thức up samples
    - chỉ up 2 nhãn mất cân bằng
        - image = multi-img + no-text
        - text = multi-text + no-image

# up samples

## utils

In [1]:
import pandas as pd
import json
import numpy as np
import random
import matplotlib.pyplot as plt
from PIL import Image

def plot_df(new_df, i1, i2):
    # Select the subset of the DataFrame between indices i1 and i2
    subset = new_df.iloc[i1:i2+1]
    for _, row in subset.iterrows():
        # Load and display the image with the specified path prefix
        img_path = '/teamspace/studios/uit-fine-tuning/data/train-images/' + row['image']
        image = Image.open(img_path).convert('RGB')
        
        # Set up a new figure for each image
        plt.figure(figsize=(5, 5))
        plt.imshow(image)
        plt.axis('off')  # Hide the axis for a cleaner look
        
        # Set the title as caption and label
        plt.title(f"Caption: {row['caption']}\nLabel: {row['label']}", fontsize=12)
        
        # Display the figure
        plt.show()

def df_to_json(merge_df, output_path = 'new_data.json'):
    json_data = merge_df.to_json(orient='index', force_ascii=False)

    # Load the JSON string into a Python dictionary
    json_dict = json.loads(json_data)

    # Convert the dictionary back to a JSON string with pretty print
    formatted_json = json.dumps(json_dict, indent=4, ensure_ascii=False)

    # Print the pretty-printed JSON
    # print(formatted_json)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(formatted_json)

## read

In [2]:
import pandas as pd
import json
import numpy as np

with open('/teamspace/studios/uit-fine-tuning/data/vimmsd-train.json') as f:
    data = json.load(f)
df = pd.DataFrame(data).T
# df.head()
print(f"{df.shape=}")
print(df['label'].value_counts())

df.shape=(10805, 3)
label
not-sarcasm      6062
multi-sarcasm    4224
image-sarcasm     442
text-sarcasm       77
Name: count, dtype: int64


In [3]:
################TUNE THIS##################
n_for_image = 2000 - 442
n_for_text =  3000 -77
#######################################
print(f"we need {n_for_image} samples more for image")
print(f"we need {n_for_text} samples more for text")
print('-'*50)
multi_sarcasm_indices = df[df['label'] == 'multi-sarcasm'].index.values
not_sarcasm_indices = df[df['label'] == 'not-sarcasm'].index.values
print(f"{len(multi_sarcasm_indices)=}")
print(f"{len(not_sarcasm_indices)=}")

we need 1558 samples more for image
we need 2923 samples more for text
--------------------------------------------------
len(multi_sarcasm_indices)=4224
len(not_sarcasm_indices)=6062


## GET INDICES

In [4]:
print(f'We need {n_for_image} samples for image-sarcasm')
# image-only = multi-img + no-text
image_sarcasm_img_indices = np.random.choice(
    multi_sarcasm_indices,
    size = n_for_image,
    replace=True
)
image_sarcasm_text_indices = np.random.choice(
    not_sarcasm_indices,
    size = n_for_image,
    replace=True
)

print(len(image_sarcasm_img_indices))
print(len(image_sarcasm_text_indices))

##################
print(f'We need {n_for_text} samples for text-sarcasm')
# text = multi-text + no-img
text_sarcasm_img_indices = np.random.choice(
    not_sarcasm_indices,
    size = n_for_text,
    replace=True
)
text_sarcasm_text_indices = np.random.choice(
    multi_sarcasm_indices,
    size = n_for_text,
    replace=True
)

print(len(text_sarcasm_img_indices))
print(len(text_sarcasm_text_indices))

We need 1558 samples for image-sarcasm
1558
1558
We need 2923 samples for text-sarcasm
2923
2923


## NEW DF

In [5]:
#################TEXT#############3
# new df for extra text-sarcasm
text_sarcasm_label = ['text-sarcasm'] * n_for_text
# convert 2 np arr?
# text_sarcasm_labels = np.
text_sarcasm_image = df.loc[text_sarcasm_img_indices, 'image'].values
text_sarcasm_text = df.loc[text_sarcasm_text_indices, 'caption'].values
print(
    len(text_sarcasm_label),
    len(text_sarcasm_image),
    len(text_sarcasm_text)
)
text_sarcasm_df = pd.DataFrame(
    {'image':text_sarcasm_image,
    'caption': text_sarcasm_text,
    'label' : text_sarcasm_label}
)
print(text_sarcasm_df['label'].value_counts())


2923 2923 2923
label
text-sarcasm    2923
Name: count, dtype: int64


In [6]:
# new df for extra text-sarcasm
image_sarcasm_label = ['image-sarcasm'] * n_for_image
image_sarcasm_image = df.loc[image_sarcasm_img_indices, 'image'].values
image_sarcasm_text = df.loc[image_sarcasm_text_indices, 'caption'].values

### NEW
## thêm phần vô tri, để tập trung vào ảnh 
# import random
# import numpy as np 
# choices = ['ý tôi là', 'no cap', 'không nói nhiều']
# image_sarcasm_text = np.random.choice(choices, size=n_for_image).tolist()

print(
    len(image_sarcasm_label),
    len(image_sarcasm_image),
    len(image_sarcasm_text)
)

image_sarcasm_df = pd.DataFrame(
    {'image':image_sarcasm_image,
    'caption': image_sarcasm_text,
    'label' : image_sarcasm_label}
)
image_sarcasm_df['label'].value_counts()


1558 1558 1558


label
image-sarcasm    1558
Name: count, dtype: int64

## merge

In [7]:
merge_df = pd.concat([df, text_sarcasm_df, image_sarcasm_df], ignore_index=True,sort=False)
print(merge_df.shape)
print(merge_df.columns)
print(merge_df['label'].value_counts())

(15286, 3)
Index(['image', 'caption', 'label'], dtype='object')
label
not-sarcasm      6062
multi-sarcasm    4224
text-sarcasm     3000
image-sarcasm    2000
Name: count, dtype: int64


In [8]:
df_to_json(merge_df, 'data_23.json')